# domain_qna Self-Retrieval 평가

qna.parquet의 질문을 쿼리로 넣어 정답 행이 Top-K 안에 포함되는지 측정.

**메트릭**
- `Recall@K`: 정답이 Top-K 안에 있는 비율
- `MRR` (Mean Reciprocal Rank): 정답이 몇 위에 나오는지 평균

**평가 조건**
- 필터 없음 (no filter)
- species 필터 적용
- species + category 필터 적용 (실제 LangGraph 조건)

## 0. 초기화

In [1]:
import os
import sys
sys.path.insert(0, "../../..")

import pandas as pd
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv("../../../infra/.env")

from qdrant_client import QdrantClient
from qdrant_client.models import (
    FieldCondition, Filter, Fusion, FusionQuery,
    MatchAny, MatchValue, Prefetch, SparseVector,
)
from fastembed import SparseTextEmbedding, TextEmbedding

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
client = QdrantClient(url=QDRANT_URL)

QNA_PATH = Path("../../../output/domain/qna.parquet")
df = pd.read_parquet(QNA_PATH)
print(f"qna.parquet: {len(df):,}행")
df.head(3)

/opt/anaconda3/envs/final-project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qna.parquet: 2,411행


,no,species,source,category,question,answer,notes
0,1,cat,bemypet,건강 및 질병,고양이 '췌장염'이 재발하기 쉬운 이유는 무엇인가요?,"고양이 췌장염은 원인이 불분명한 경우가 많고, 염증성 장질환(IBD)이나 간 질환이...",ISFM Guidelines; 췌장염은 완치보다 장기적인 관리가 중요한 질환임 =I...
1,2,cat,bemypet,사육 및 관리,췌장염을 앓았던 고양이의 식단 관리 원칙은?,"췌장에 무리를 주지 않도록 고단백, 저지방, 소화 흡수가 용이한 사료를 선택해야 합...",Hand et al. (2010); 처방식 사료 외의 기름진 간식 급여는 절대 금지...
2,3,cat,bemypet,건강 및 질병,고양이 '당뇨병'의 대표적인 초기 증상 '3다(多)'는?,"다음(물을 많이 마심), 다뇨(소변량이 늘어남), 다식(많이 먹지만 살은 빠짐)입니...",Nelson & Couto (2019); 갑작스러운 식욕 증가와 체중 감소가 동시에...


In [2]:
print("Dense 모델 로드 중...")
dense_model = TextEmbedding("intfloat/multilingual-e5-large")
print("Sparse 모델 로드 중...")
sparse_model = SparseTextEmbedding("Qdrant/bm25")
print("완료")

Dense 모델 로드 중...


/var/folders/n6/9fr973913k7931ymsg4ly6rc0000gn/T/ipykernel_43741/2718869934.py:2: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  dense_model = TextEmbedding("intfloat/multilingual-e5-large")


Sparse 모델 로드 중...
완료


In [3]:
def embed(query: str):
    dv = list(dense_model.embed([f"query: {query}"]))[0].tolist()
    sv = list(sparse_model.embed([query]))[0]
    return dv, SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist())


def search(query: str, top_k: int = 10, qdrant_filter=None):
    dv, sv = embed(query)
    return client.query_points(
        collection_name="domain_qna",
        prefetch=[
            Prefetch(query=dv, using="dense", limit=top_k * 3, filter=qdrant_filter),
            Prefetch(query=sv, using="sparse", limit=top_k * 3, filter=qdrant_filter),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    ).points


def evaluate(sample_df: pd.DataFrame, top_k: int = 10, use_species: bool = False, use_category: bool = False):
    """
    sample_df의 각 행에 대해 question을 쿼리로 검색 후 정답(no)이 Top-K 안에 있는지 측정.
    """
    hits = {1: 0, 3: 0, 5: 0, 10: 0}
    reciprocal_ranks = []

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="평가 중"):
        must = []
        if use_species:
            species_val = row["species"]
            if species_val == "both":
                must.append(FieldCondition(key="species", match=MatchAny(any=["dog", "cat", "both"])))
            else:
                must.append(FieldCondition(key="species", match=MatchAny(any=[species_val, "both"])))
        if use_category:
            must.append(FieldCondition(key="category", match=MatchValue(value=row["category"])))
        f = Filter(must=must) if must else None

        results = search(row["question"], top_k=top_k, qdrant_filter=f)
        retrieved_nos = [p.payload["no"] for p in results]

        # Recall@K
        for k in [1, 3, 5, 10]:
            if row["no"] in retrieved_nos[:k]:
                hits[k] += 1

        # MRR
        if row["no"] in retrieved_nos:
            rank = retrieved_nos.index(row["no"]) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    n = len(sample_df)
    return {
        "Recall@1":  hits[1]  / n,
        "Recall@3":  hits[3]  / n,
        "Recall@5":  hits[5]  / n,
        "Recall@10": hits[10] / n,
        "MRR":       sum(reciprocal_ranks) / n,
        "n":         n,
    }

## 1. 샘플 추출

전체 2,411건을 다 돌리면 오래 걸리므로 species × category 층화 샘플링

In [4]:
# species × category 층화 샘플링 (각 그룹에서 최대 10건)
# pandas groupby.apply() 결과에서 group key가 MultiIndex로 올라가는 버전 이슈 방지 → pd.concat 사용
parts = [
    g.sample(min(len(g), 10), random_state=42)
    for _, g in df.groupby(["species", "category"])
]
sample = pd.concat(parts).reset_index(drop=True)
print(f"샘플: {len(sample)}건")
print(sample.groupby(["species", "category"]).size().to_string())

샘플: 107건
species  category
both     건강 및 질병      4
         사육 및 관리     10
         여행 및 이동      8
         영양 및 식단      5
cat      건강 및 질병     10
         사육 및 관리     10
         영양 및 식단     10
         행동 및 심리     10
dog      건강 및 질병     10
         사육 및 관리     10
         영양 및 식단     10
         행동 및 심리     10


## 2. 조건별 평가

In [7]:
# 필터 없음
r1 = evaluate(sample, use_species=False, use_category=False)
print("[필터 없음]", r1)

평가 중: 100%|██████████| 107/107 [00:07<00:00, 14.50it/s]

[필터 없음] {'Recall@1': 0.9345794392523364, 'Recall@3': 0.9906542056074766, 'Recall@5': 1.0, 'Recall@10': 1.0, 'MRR': 0.9649532710280374, 'n': 107}


In [8]:
# species 필터만
r2 = evaluate(sample, use_species=True, use_category=False)
print("[species 필터]", r2)

평가 중: 100%|██████████| 107/107 [00:06<00:00, 15.84it/s]

[species 필터] {'Recall@1': 0.9439252336448598, 'Recall@3': 0.9813084112149533, 'Recall@5': 1.0, 'Recall@10': 1.0, 'MRR': 0.9663551401869159, 'n': 107}


In [9]:
# species + category 필터 (실제 LangGraph 조건)
r3 = evaluate(sample, use_species=True, use_category=True)
print("[species + category 필터]", r3)

평가 중: 100%|██████████| 107/107 [00:07<00:00, 15.23it/s]

[species + category 필터] {'Recall@1': 0.9252336448598131, 'Recall@3': 0.9813084112149533, 'Recall@5': 1.0, 'Recall@10': 1.0, 'MRR': 0.9559190031152648, 'n': 107}


## 3. 결과 비교

In [10]:
result_df = pd.DataFrame([
    {"조건": "필터 없음",            **r1},
    {"조건": "species 필터",         **r2},
    {"조건": "species + category",   **r3},
]).set_index("조건")

display(result_df.style.format({
    "Recall@1":  "{:.1%}",
    "Recall@3":  "{:.1%}",
    "Recall@5":  "{:.1%}",
    "Recall@10": "{:.1%}",
    "MRR":       "{:.3f}",
    "n":         "{:.0f}",
}))

,Recall@1,Recall@3,Recall@5,Recall@10,MRR,n
조건,,,,,,
필터 없음,93.5%,99.1%,100.0%,100.0%,0.965,107
species 필터,94.4%,98.1%,100.0%,100.0%,0.966,107
species + category,92.5%,98.1%,100.0%,100.0%,0.956,107


## 4. 실패 케이스 분석

In [11]:
# species + category 필터 조건에서 Top-5 안에 못 들어온 케이스
failures = []

for _, row in sample.iterrows():
    must = [
        FieldCondition(key="species", match=MatchAny(any=[row["species"], "both"])),
        FieldCondition(key="category", match=MatchValue(value=row["category"])),
    ]
    results = search(row["question"], top_k=5, qdrant_filter=Filter(must=must))
    retrieved_nos = [p.payload["no"] for p in results]

    if row["no"] not in retrieved_nos:
        failures.append({
            "no":       row["no"],
            "species":  row["species"],
            "category": row["category"],
            "question": row["question"],
            "top1_q":   results[0].payload.get("question") if results else None,
        })

print(f"실패 케이스: {len(failures)}건 / {len(sample)}건")
pd.DataFrame(failures)

실패 케이스: 0건 / 107건


""
